# Lecture : Shallow Graph Learning

## Lab 04 : DeepWalk - Exercise

### Xavier Bresson, Guoji Fu   

Perozzi, Al-Rfou, Skiena, DeepWalk: Online learning of social representations, 2014  
https://arxiv.org/pdf/1403.6652.pdf

Notebook goals :<br>
• Design a random walk extractor <br>
• Implement the deepwalk technique <br>
• Compare visually the deepwalk embedding with networkx visualization <br>


In [ ]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/05_Shallow_Learning'
    print(path_to_file)
    # change current path to the folder containing "path_to_file"
    os.chdir(path_to_file)
    !pwd
    !pip install rdkit==2026.3.5 # Install RDKit
    !pip -q install -f https://data.dgl.ai/wheels/torch-2.6/repo.html dgl==2.5.0 # Install DGL


In [ ]:
# Libraries
import pickle
import sys; sys.path.insert(0, 'lib/')
from lib.utils import Molecule
from rdkit import Chem
import torch
import torch.nn as nn
import networkx as nx
import matplotlib.pyplot as plt
import random
from lib.utils import compute_ncut


## Load dataset and select one molecule

In [ ]:
print('Loading data')
data_folder_pytorch = 'datasets/ZINC_pytorch/'
with open(data_folder_pytorch+"train_pytorch.pkl","rb") as f:
    dataset=pickle.load(f)

# Select one molecule
idx = 12
mol = dataset[idx]
print(mol.atom_type)
print(mol.atom_type_pe)
print(mol.bond_type)
print(mol.bag_of_atoms)
print(mol.logP_SA_cycle_normalized)
print(mol.smile)
Chem.MolFromSmiles(mol.smile)


## Exercise 1 : Design a random walk extractor

### Question 1.1 : Implement a class that generates a random walk path.

Hints:
- Sample the next node from the RW probability `prob_j = RW[i,:]`.
- Sample from the categorical distribution with `torch.distributions.Categorical(prob_j).sample()`.


In [ ]:
class sample_RW_path:

    def __init__(self, num_steps, RW):
        self.num_steps = num_steps # number of steps
        self.RW = RW # random walk matrix
        self.num_nodes = RW.size(0) # number of nodes

    def sample_walk(self, idx_start):
        idx = torch.tensor(idx_start).long() # starting index of the walk
        RWpath = [idx] # random walk path
        for _ in range(self.num_steps-1):
            ########################################
            # YOUR CODE START
            # sample the next node from the categorical distribution RW[i,:]
            # use torch.distributions.Categorical(prob).sample()
            ########################################

            idx = 

            ########################################
            # YOUR CODE END
            ########################################

            RWpath.append(idx) # append sampled node to the path
        RWpath = torch.stack(RWpath).flatten() # path format = torch.tensor([idx_1, idx_2, ..., idx_num_steps])
        return RWpath



### Question 1.2 : Use the previously defined function to sample a random walk path.

Hints:
- Step 1: Compute the RW operator with $RW = D^{-1}A$.
- Step 2: Choose the number of RW steps.
- Step 3: Use `sample_RW_path()` to instantiate a RW class.
- Step 4: Apply `sample_walk()` to extract a RW path.
  

In [ ]:
A = (mol.bond_type>0).float() # Adjacency matrix
D = A.sum(dim=1) # Degree vector (row degrees)

########################################
# YOUR CODE START
########################################

# Step 1: # Compute the RW operator
# Compute the inverse degree matrix
Dinv = 

# Compute RW matrix
RW = 

# Step 2: choose the number of RW steps
num_RW_steps = T = 

# Step 3: instantiate RW class
generator = 

# Step 4: sample RW path starting with index=7
walk = 

########################################
# YOUR CODE END
########################################

print('RW:',walk)

# Visualise the RW
fig = plt.figure()
ax = fig.add_subplot(111)
A_nx = nx.from_numpy_array(A.numpy())
# Color random-walk nodes
walk_nodes = set(walk.tolist())
node_color = ['orange' if i in walk_nodes else 'lightblue' for i in range(A.size(0))]
nx.draw(A_nx, ax=ax, node_color=node_color, with_labels=True, font_size=10)
ax.title.set_text(f'Molecule and random walk {walk.tolist()} visualization with networkx')
plt.show()


## Exercise 2 : Implement DeepWalk-style node embeddings with Skip-gram negative sampling

For pedagogical simplicity, this exercise uses a **single (tied) node-embedding table** for both center and context nodes. The original DeepWalk paper uses Skip-gram with hierarchical softmax.


### Question 2.1 : Implement a DeepWalk-style network and apply it to molecular graphs.

For a sampled walk $R=(v_1,\ldots,v_T)$, each occurrence $v_t$ is used as a center node, and its positive context contains the walk occurrences within a window of radius $w$ around position $t$.

Instructions:
- Step 1: Extract the center embedding $h_i$ for node occurrence $i=v_t$.
- Step 2: Extract the embeddings $h_j$ for context occurrences $j\in C_t=(v_{\max(1,t-w)},\ldots,v_{t-1},v_{t+1},\ldots,v_{\min(T,t+w)})$, \
  with $w$ being the window size (do not use a set: repeated occurrences matter).
- Step 3: For each positive pair $(i,j)$, sample `num_negative` negative nodes $k_r\sim q$, where $q(k)=1/|V|$ is uniform over the graph nodes.
- Step 4: Use the negative-sampling loss
  $$\ell(i,j)=-\log\sigma(h_i^T h_j)-\sum_{r=1}^{K}\log\sigma(-h_i^T h_{k_r}).$$


In [ ]:
class deepwalk_net(nn.Module):

    def __init__(self, num_nodes, hidden_dim, num_negative, window_size):
        super(deepwalk_net, self).__init__()
        print(num_nodes, hidden_dim)
        self.num_nodes = num_nodes
        self.num_negative = num_negative
        self.window_size = window_size
        self.node_embedding = nn.Embedding(num_nodes, hidden_dim)

    def forward(self, walk):
        loss = []

        # Each POSITION t in the walk is a center occurrence.
        # Repeated node IDs at different positions are therefore kept.
        for t in range(walk.numel()):
            i = walk[t]
        
            ########################################
            # YOUR CODE START
            ########################################
        
            # Step 1: extract center embedding h_i
            hi =               # size=(1,d)
        
            # Step 2: context occurrences within a window around position t
            context =          # size=(C,), with C <= 2w
            hj =               # size=(C,d)
            positive_scores =  # Positive scores h_i^T h_j, size=(C,1) = (C,d) x (d,1)
        
            # Step 3: sample K negatives independently for each positive pair
            #         q(k) = 1/|V|, i.e. uniform over all graph nodes
            k =                # size=(C,K)
            hk =               # size=(C,K,d)
            negative_scores =  # size=(C,K)
        
            ########################################
            # YOUR CODE END
            ########################################
        
            # Step 4: Skip-gram negative-sampling loss
            #         l(i,j) = -log sigma(h_i^T h_j)
            #                  -sum_r log sigma(-h_i^T h_{k_r})
            loss_pairs = - torch.nn.functional.logsigmoid(positive_scores.squeeze(-1)) \
                         - torch.nn.functional.logsigmoid(-negative_scores).sum(dim=1)  # size=(C,)
            loss.append(loss_pairs)
        
        return torch.cat(loss).mean()

        

### Question 2.2 : Instantiate a DeepWalk-style network

Instructions:
- Select the context-window radius and the number of negative samples.
- Instantiate `deepwalk_net()`.
- Evaluate and compare the embeddings obtained with different context-window sizes and numbers of negative samples.


In [ ]:
num_nodes = A.size(0)

########################################
# YOUR CODE START
########################################

# Hyperparameters
hidden_dim = 2     # 2D only for visualization in this lab
window_size =      # context-window radius along the random walk
num_negative =     # number of negative samples per positive pair

net = 

########################################
# YOUR CODE END
########################################

print(net)


### Train the DeepWalk-style network


In [ ]:
# Train the network
optimizer = torch.optim.Adam(net.parameters(), lr=0.003)
for iter in range(400): 
    loss_epoch = 0.0
    for idx in torch.randperm(num_nodes).tolist(): # shuffle ordering of starting nodes
        walk = generator.sample_walk(idx)
        loss = net(walk)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_epoch += loss.detach()

    # Do not recenter the embeddings here: dot-product Skip-gram losses are not translation invariant.

    # plot the loss value
    if not iter % 10:
        print(iter, loss_epoch / num_nodes)


### Visualize the learned node embeddings in 2D


In [ ]:
# Visualize the 2D coordinates of the node embeddings
x = net.node_embedding.weight.detach()
print(x,x.size())

# plot 2D coordinates
fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(x[:,0], x[:,1])
idx = list(range(num_nodes))
C = compute_ncut(A.long(), 4)
ax.scatter(x[:,0], x[:,1], c=C, cmap='jet')
for i, txt in enumerate(idx):
    ax.annotate(txt, (x[:,0][i], x[:,1][i]), textcoords="offset points", xytext=(1,5))
ax.title.set_text('2D embedding of nodes')
plt.show()

### Question 2.3 : Compare visually the learned embedding with the NetworkX visualization


In [ ]:
# Compare with graph edges
fig = plt.figure()
ax = fig.add_subplot(111)
nx.draw(A_nx, ax=ax, node_color=C, cmap='jet', with_labels=True, font_size=10) # visualise node indexes
ax.title.set_text('Molecular graph')
plt.show()
